# release16_full_probe_failure_attribution_v1

Failure attribution for `release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1` after the full walk-forward probe.

The prior full probe established `full_probe_failed_no_actor_retention`; this notebook does not create or retain actors. It classifies the failed windows, compares against the converted SOXLBacktest qlib reference and current-best YTD floor, and turns the evidence into the next admissible training objective handoff.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROFILE = "release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1"
EXP = "release16_full_probe_failure_attribution_v1"
ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from src.soxl_quant.app.oos_diagnostics import (
    aggregate_feature_differences,
    aggregate_poor_windows,
    compute_feature_differences,
    find_poor_windows,
    load_qlib_feature_frame,
)
from src.soxl_quant.app.state_conditioned_objective import (
    StateConditionedObjectiveConfig,
    daily_path_metrics,
    max_drawdown_interval_metrics,
    score_state_conditioned_objective,
    summarize_state_conditioned_objective,
)
from src.soxl_quant.app.training_objective_handoff import (
    derive_training_objective_handoff,
    render_training_objective_handoff,
)

CARD = ROOT / "runtime" / "model_selection" / "card" / EXP
MODEL_CARD = ROOT / "runtime" / "model" / "card" / PROFILE
REWARD_CARD = ROOT / "runtime" / "reward" / "card" / PROFILE
REFERENCE_CARD = ROOT / "runtime" / "production" / "card" / "soxl_backtest_policy_reimplementation_v1"
CURRENT_COMP = ROOT / "runtime" / "production" / "card" / "current_model_ytd_june_comparison_v1"

PATHS = {
    "fixed": MODEL_CARD / f"{PROFILE}_soxl_daily.csv",
    "wf_2020": MODEL_CARD / f"{PROFILE}_wf_2020_soxl_daily.csv",
    "wf_2022": MODEL_CARD / f"{PROFILE}_wf_2022_soxl_daily.csv",
    "wf_2023_2024": MODEL_CARD / f"{PROFILE}_wf_2023_2024_soxl_daily.csv",
    "full_probe_summary": REWARD_CARD / "summary.json",
    "soxlbacktest_daily": REFERENCE_CARD / "new_policy_daily.csv",
    "soxlbacktest_summary": REFERENCE_CARD / "summary.json",
    "current_best_metrics": CURRENT_COMP / "all_model_comparison_metrics.csv",
}
for label, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"{label}: {path}")

def read_json(path: Path) -> dict:
    with path.open() as fh:
        return json.load(fh)

def load_daily(path: Path) -> pd.DataFrame:
    daily = pd.read_csv(path)
    if "date" in daily.columns and "datetime" not in daily.columns:
        daily = daily.rename(columns={"date": "datetime"})
    daily["datetime"] = pd.to_datetime(daily["datetime"])
    return daily.sort_values("datetime").reset_index(drop=True)

fixed_daily = load_daily(PATHS["fixed"])
wf_daily = {name: load_daily(path) for name, path in PATHS.items() if name.startswith("wf_")}
reference_daily_raw = load_daily(PATHS["soxlbacktest_daily"])
reference_ytd = reference_daily_raw[reference_daily_raw["window"].eq("ytd_2026")].copy()
reference_june = reference_daily_raw[reference_daily_raw["window"].eq("june_2026")].copy()
full_probe_summary = read_json(PATHS["full_probe_summary"])
soxlbacktest_summary = read_json(PATHS["soxlbacktest_summary"])
current_best_metrics = pd.read_csv(PATHS["current_best_metrics"])

artifact_context = {
    "profile": PROFILE,
    "experiment": EXP,
    "full_probe_decision": full_probe_summary["decision"],
    "full_probe_failed_gates": full_probe_summary["failed_gates"],
    "soxlbacktest_ytd_return_pct": soxlbacktest_summary["windows"]["ytd_2026"]["new_policy"]["rulebase_return_pct"],
    "soxlbacktest_ytd_max_drawdown_pct": soxlbacktest_summary["windows"]["ytd_2026"]["new_policy"]["rulebase_max_drawdown_pct"],
    "soxlbacktest_alignment_delta_pct": soxlbacktest_summary["windows"]["ytd_2026"]["comparison"]["return_delta_pct"],
}
artifact_context


{'profile': 'release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1',
 'experiment': 'release16_full_probe_failure_attribution_v1',
 'full_probe_decision': 'full_probe_failed_no_actor_retention',
 'full_probe_failed_gates': ['fixed_test_max_drawdown',
  'fixed_promotion_eligible',
  'actor_retention_allowed',
  'ytd_return_vs_soxlbacktest_reference',
  'ytd_return_vs_current_best_floor',
  'wf_2020_relative_outperformance',
  'wf_2022_return_nonnegative',
  'wf_2022_sharpe_nonnegative',
  'wf_2023_2024_return_nonnegative',
  'wf_2023_2024_max_drawdown',
  'wf_2023_2024_relative_outperformance',
  'wf_2023_2024_sharpe_nonnegative'],
 'soxlbacktest_ytd_return_pct': 132.07797795135497,
 'soxlbacktest_ytd_max_drawdown_pct': 13.554088542803521,
 'soxlbacktest_alignment_delta_pct': 0.0}

In [2]:
def slice_by_year(daily: pd.DataFrame, year: int) -> pd.DataFrame:
    return daily[pd.to_datetime(daily["datetime"]).dt.year == year].copy()

def row_metrics(label: str, daily: pd.DataFrame, *, source: str) -> dict[str, object]:
    m = daily_path_metrics(daily)
    dd = max_drawdown_interval_metrics(daily)
    return {
        "window": label,
        "source": source,
        "start": m["start"],
        "end": m["end"],
        "rows": m["rows"],
        "return_pct": m["return"],
        "buyhold_return_pct": m["buy_and_hold_return"],
        "relative_return_pp": m["relative_return"],
        "max_drawdown_pct": m["max_dd"],
        "avg_position": m["avg_pos"],
        "drawdown_peak": dd["peak_date"],
        "drawdown_trough": dd["trough_date"],
        "drawdown_interval_avg_pos": dd["interval_avg_pos"],
        "drawdown_interval_trades": dd["interval_trades"],
    }

metrics_rows = [
    row_metrics("fixed_2020_2026", fixed_daily, source="release16_full_probe"),
    row_metrics("ytd_2026", slice_by_year(fixed_daily, 2026), source="release16_full_probe"),
    row_metrics("ytd_2026", reference_ytd, source="soxlbacktest_qlib_reference"),
    row_metrics("june_2026", reference_june, source="soxlbacktest_qlib_reference"),
]
for name, daily in wf_daily.items():
    metrics_rows.append(row_metrics(name, daily, source="release16_full_probe"))
window_metrics = pd.DataFrame(metrics_rows)

ytd_leaders = (
    current_best_metrics[current_best_metrics["window"].eq("ytd_2026_common")]
    .sort_values("current_model_return_pct", ascending=False)
    .head(8)
    .reset_index(drop=True)
)
current_best_floor = 213.143462
reference_return = float(artifact_context["soxlbacktest_ytd_return_pct"])
release16_ytd = window_metrics[(window_metrics["source"].eq("release16_full_probe")) & (window_metrics["window"].eq("ytd_2026"))].iloc[0]
release16_vs_baselines = pd.DataFrame(
    [
        {
            "gate": "soxlbacktest_reference_return",
            "actual_return_pct": release16_ytd["return_pct"],
            "required_return_pct": reference_return,
            "gap_pct": release16_ytd["return_pct"] - reference_return,
            "passed": release16_ytd["return_pct"] >= reference_return,
        },
        {
            "gate": "current_best_alignment_floor",
            "actual_return_pct": release16_ytd["return_pct"],
            "required_return_pct": current_best_floor,
            "gap_pct": release16_ytd["return_pct"] - current_best_floor,
            "passed": release16_ytd["return_pct"] >= current_best_floor,
        },
        {
            "gate": "ytd_max_drawdown_lt_20",
            "actual_return_pct": release16_ytd["max_drawdown_pct"],
            "required_return_pct": 20.0,
            "gap_pct": 20.0 - release16_ytd["max_drawdown_pct"],
            "passed": release16_ytd["max_drawdown_pct"] < 20.0,
        },
    ]
)
window_metrics


,window,source,start,end,rows,return_pct,buyhold_return_pct,relative_return_pp,max_drawdown_pct,avg_position,drawdown_peak,drawdown_trough,drawdown_interval_avg_pos,drawdown_interval_trades
0,fixed_2020_2026,release16_full_probe,2020-01-02,2026-06-09,1617,291.034886,936.169263,-645.134377,25.329680,0.195414,2021-12-27,2022-12-28,0.148564,31
1,ytd_2026,release16_full_probe,2026-01-02,2026-06-09,109,38.555494,326.926303,-288.370809,9.559553,0.085256,2026-02-25,2026-03-30,0.091250,5
2,ytd_2026,soxlbacktest_qlib_reference,2026-01-02,2026-06-09,109,132.077978,326.926318,-194.848340,13.554089,0.162207,2026-05-11,2026-05-18,0.535900,0
3,june_2026,soxlbacktest_qlib_reference,2026-06-01,2026-06-09,7,3.138960,-11.165925,14.304885,1.528209,0.128314,2026-06-08,2026-06-09,0.299400,0
4,wf_2020,release16_full_probe,2020-01-02,2020-12-30,252,35.374744,58.093571,-22.718827,19.260466,0.184746,2020-09-02,2020-09-08,0.204190,1
5,wf_2022,release16_full_probe,2022-01-03,2022-12-29,250,-16.467158,-86.588072,70.120914,18.149535,0.090647,2022-08-04,2022-12-28,0.084184,19
6,wf_2023_2024,release16_full_probe,2024-01-02,2024-12-30,251,-7.690929,-0.356635,-7.334294,20.914378,0.167797,2024-07-10,2024-12-19,0.117520,18


In [3]:
windows_by_profile = {}
feature_diffs_by_profile = {}
source_daily_by_label = {
    "release16_fixed": fixed_daily,
    "release16_ytd": slice_by_year(fixed_daily, 2026),
    "release16_wf_2020": wf_daily["wf_2020"],
    "release16_wf_2022": wf_daily["wf_2022"],
    "release16_wf_2023_2024": wf_daily["wf_2023_2024"],
}

all_start = min(d["datetime"].min() for d in source_daily_by_label.values()).date().isoformat()
all_end = max(d["datetime"].max() for d in source_daily_by_label.values()).date().isoformat()
feature_frame = load_qlib_feature_frame(all_start, all_end)

for label, daily in source_daily_by_label.items():
    size = 21 if len(daily) < 126 else 63
    windows = find_poor_windows(
        daily,
        window_size=size,
        top_n=12,
        target_excess=30.0,
        target_max_dd=15.0,
    )
    windows["diagnostic_label"] = label
    windows["window_size"] = size
    windows_by_profile[label] = windows
    feature_diffs_by_profile[label] = compute_feature_differences(
        feature_frame,
        windows,
        daily=daily,
        top_n=30,
    )

poor_windows = pd.concat(windows_by_profile.values(), ignore_index=True)
poor_window_clusters = aggregate_poor_windows(
    windows_by_profile,
    target_excess=30.0,
    target_max_dd=15.0,
    overlap_gap_days=7,
)
feature_differences = aggregate_feature_differences(feature_diffs_by_profile, top_n=50)

FEATURE_GROUP_RULES = [
    ("trend_quality", ("anchor", "trend", "ma", "slope", "momentum", "mom_", "mom")),
    ("relative_strength_spread", ("relative_strength", "soxx", "spread", "ratio")),
    ("volatility_regime", ("vix", "vol", "rv", "realized", "hv", "downside")),
    ("drawdown_path", ("drawdown", "cum_drop", "below_anchor", "days_since", "loss_streak")),
    ("path_decay_or_tracking", ("tracking", "decay", "jump", "variation")),
    ("trade_state", ("position", "turnover", "trade", "action")),
]

def feature_group(feature: str) -> str:
    lower = feature.lower()
    for group, tokens in FEATURE_GROUP_RULES:
        if any(token in lower for token in tokens):
            return group
    return "other"

if feature_differences.empty:
    feature_group_consensus = pd.DataFrame(columns=["feature_group", "top_mentions", "mean_abs_delta_z", "mean_delta_z", "example_features"])
else:
    tmp = feature_differences.copy()
    tmp["feature_group"] = tmp["feature"].map(feature_group)
    tmp["abs_delta_z"] = tmp["mean_abs_delta_z"].abs() if "mean_abs_delta_z" in tmp.columns else tmp["mean_delta_z"].abs()
    group_rows = []
    for group, sub in tmp.groupby("feature_group"):
        ranked = sub.sort_values("abs_delta_z", ascending=False)
        group_rows.append(
            {
                "feature_group": group,
                "top_mentions": int(len(sub)),
                "mean_abs_delta_z": float(ranked["abs_delta_z"].mean()),
                "mean_delta_z": float(ranked["mean_delta_z"].mean()) if "mean_delta_z" in ranked.columns else float("nan"),
                "example_features": ";".join(ranked["feature"].head(5).astype(str).tolist()),
            }
        )
    feature_group_consensus = pd.DataFrame(group_rows).sort_values(
        ["top_mentions", "mean_abs_delta_z", "feature_group"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

poor_window_clusters.head(10)


[39367:MainThread](2026-06-15 02:32:29,031) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[39367:MainThread](2026-06-15 02:32:29,450) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[39367:MainThread](2026-06-15 02:32:29,450) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[39367:MainThread](2026-06-15 02:32:30,779) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[39367:MainThread](2026-06-15 02:32:30,779) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[39367:MainThread](2026-06-15 02:32:30,780) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C006,2026-02-26,2026-06-09,2,22,"release16_fixed,release16_ytd",-197.394389,-409.721765,4.396998,7.732725,0.007496,227.394389,439.721765,underparticipation,reentry_participation_or_behavior_clone
1,C005,2025-04-08,2025-07-21,1,2,release16_fixed,-205.936552,-210.533292,2.013101,2.013101,0.346590,235.936552,240.533292,underparticipation,reentry_participation_or_behavior_clone
2,C001,2020-03-16,2020-08-13,1,12,release16_wf_2020,-126.172370,-181.417852,4.111637,4.270351,0.285191,156.172370,211.417852,underparticipation,reentry_participation_or_behavior_clone
3,C004,2024-01-02,2024-07-22,1,12,release16_wf_2023_2024,-58.155635,-76.386938,2.693744,9.403118,0.266415,88.155635,106.386938,underparticipation,reentry_participation_or_behavior_clone
4,C003,2022-09-01,2022-12-29,1,11,release16_wf_2022,-7.004310,-13.758067,9.000394,11.821636,0.038139,37.004310,43.758067,underparticipation,reentry_participation_or_behavior_clone
5,C002,2022-05-11,2022-08-10,1,1,release16_wf_2022,-2.402002,-2.402002,5.832322,5.832322,0.129021,32.402002,32.402002,underparticipation,reentry_participation_or_behavior_clone


In [4]:
metrics_by_candidate = {
    "release16_full_probe": {
        "ytd_2026": daily_path_metrics(slice_by_year(fixed_daily, 2026)),
        "wf_2020": daily_path_metrics(wf_daily["wf_2020"]),
        "wf_2022": daily_path_metrics(wf_daily["wf_2022"]),
        "wf_2023_2024": daily_path_metrics(wf_daily["wf_2023_2024"]),
    },
}
cluster_metrics_by_candidate = {
    "release16_full_probe": {
        row.cluster_id: {
            "relative_return": row.mean_excess_return,
            "max_dd": row.max_max_dd,
            "avg_pos": row.mean_avg_pos,
        }
        for row in poor_window_clusters.itertuples(index=False)
    }
}
objective_scores = score_state_conditioned_objective(
    metrics_by_candidate,
    cluster_metrics_by_candidate=cluster_metrics_by_candidate,
    config=StateConditionedObjectiveConfig(target_relative_return=30.0, target_max_dd=15.0),
)
objective_summary = summarize_state_conditioned_objective(objective_scores)
handoff = derive_training_objective_handoff(
    walkforward_scores=objective_scores,
    poor_window_clusters=poor_window_clusters,
    feature_groups=feature_group_consensus,
)
objective_scores


,candidate,window_count,window_pass_count,window_gate,cluster_count,cluster_gate,objective_pass,objective_loss,relative_loss,drawdown_loss,participation_loss,min_relative_return,max_max_dd,mean_avg_pos,worst_window,worst_cluster
0,release16_full_probe,4,0,False,6,False,False,1040.322519,796.490173,141.117632,102.714714,-288.370809,20.914378,0.132112,ytd_2026,C006


In [5]:
CARD.mkdir(parents=True, exist_ok=True)
window_metrics.to_csv(CARD / "window_metrics.csv", index=False)
ytd_leaders.to_csv(CARD / "current_best_ytd_leaders.csv", index=False)
release16_vs_baselines.to_csv(CARD / "release16_vs_baseline_gates.csv", index=False)
poor_windows.to_csv(CARD / "poor_windows.csv", index=False)
poor_window_clusters.to_csv(CARD / "poor_window_clusters.csv", index=False)
feature_differences.to_csv(CARD / "feature_differences.csv", index=False)
feature_group_consensus.to_csv(CARD / "feature_group_consensus.csv", index=False)
objective_scores.to_csv(CARD / "objective_scores.csv", index=False)
(CARD / "training_objective_handoff.md").write_text(render_training_objective_handoff(handoff), encoding="utf-8")
(CARD / "training_objective_handoff.json").write_text(json.dumps(handoff, indent=2) + "\n", encoding="utf-8")
selection = {
    "experiment": EXP,
    "profile": PROFILE,
    "decision": "release16_full_probe_failure_attribution_no_actor_retention",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "selected_next_mechanism": handoff.get("decision"),
    "objective_summary": objective_summary,
    "handoff": handoff,
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2) + "\n", encoding="utf-8")
summary = {
    "experiment": EXP,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "profile": PROFILE,
    "decision": "release16_full_probe_failure_attribution_no_actor_retention",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "soxlbacktest_reference": {
        "converted_qilib_daily_path": str(PATHS["soxlbacktest_daily"].relative_to(ROOT)),
        "ytd_return_pct": float(reference_return),
        "ytd_max_drawdown_pct": float(artifact_context["soxlbacktest_ytd_max_drawdown_pct"]),
        "legacy_alignment_delta_pct": float(artifact_context["soxlbacktest_alignment_delta_pct"]),
    },
    "release16_ytd": {
        "return_pct": float(release16_ytd["return_pct"]),
        "relative_return_pp": float(release16_ytd["relative_return_pp"]),
        "max_drawdown_pct": float(release16_ytd["max_drawdown_pct"]),
        "avg_position": float(release16_ytd["avg_position"]),
    },
    "baseline_gates_failed": release16_vs_baselines.loc[~release16_vs_baselines["passed"], "gate"].tolist(),
    "poor_window_cluster_count": int(len(poor_window_clusters)),
    "dominant_failure_types": poor_window_clusters["dominant_failure_type"].value_counts().to_dict() if not poor_window_clusters.empty else {},
    "top_feature_groups": feature_group_consensus.head(5).to_dict(orient="records"),
    "objective_summary": objective_summary,
    "handoff": handoff,
    "sidecars": [
        "window_metrics.csv",
        "current_best_ytd_leaders.csv",
        "release16_vs_baseline_gates.csv",
        "poor_windows.csv",
        "poor_window_clusters.csv",
        "feature_differences.csv",
        "feature_group_consensus.csv",
        "objective_scores.csv",
        "training_objective_handoff.json",
        "training_objective_handoff.md",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

worst_clusters = poor_window_clusters.head(3)
cluster_lines = []
for row in worst_clusters.itertuples(index=False):
    cluster_lines.append(
        f"- `{row.cluster_id}` {row.cluster_start}..{row.cluster_end}: "
        f"type `{row.dominant_failure_type}`, mean excess {row.mean_excess_return:.1f}pp, "
        f"max MaxDD {row.max_max_dd:.1f}%, mean pos {row.mean_avg_pos:.3f}."
    )
if not cluster_lines:
    cluster_lines = ["- No poor-window clusters were produced."]
feature_lines = []
for row in feature_group_consensus.head(5).itertuples(index=False):
    feature_lines.append(f"- `{row.feature_group}`: mentions {row.top_mentions}, mean |delta_z| {row.mean_abs_delta_z:.2f}; examples {row.example_features}")
if not feature_lines:
    feature_lines = ["- No feature-group consensus was produced."]
conclusion = f"""# {EXP} conclusion\n\nDecision: `release16_full_probe_failure_attribution_no_actor_retention`.\n\nThe latest release16 full probe remains non-retainable. The converted SOXLBacktest qlib reference is still accepted as the YTD baseline: return {reference_return:.1f}%, MaxDD {artifact_context['soxlbacktest_ytd_max_drawdown_pct']:.1f}%, legacy delta {artifact_context['soxlbacktest_alignment_delta_pct']:.2g}pp. Release16 YTD return is {release16_ytd['return_pct']:.1f}% with relative {release16_ytd['relative_return_pp']:.1f}pp and avg position {release16_ytd['avg_position']:.3f}, which fails both SOXLBacktest reference and current-best return floors despite YTD MaxDD staying below 20%.\n\n## Poor-window clusters\n\n""" + "\n".join(cluster_lines) + f"""\n\n## Feature-state consensus\n\n""" + "\n".join(feature_lines) + f"""\n\n## Handoff\n\n`derive_training_objective_handoff()` returns `{handoff.get('decision')}` with objective terms: {', '.join(str(x) for x in handoff.get('objective_terms', []))}. This means the next admissible work is a state-conditioned multi-objective target/checkpoint screen, not another unchanged PPO run and not actor retention.\n\nNext action: build or reuse a fold-local objective/target screen that jointly penalizes walk-forward relative-return gaps and drawdown gaps, with explicit high-vol/rebound participation and downtrend drawdown-state control. Any follow-up training branch must still pass fixed split, `wf_2020`, `wf_2022`, `wf_2023_2024`, YTD SOXLBacktest/current-best gates, model replay export, and dashboard runnable-source checks before retention.\n"""
(CARD / "conclusion.md").write_text(conclusion, encoding="utf-8")
summary


{'experiment': 'release16_full_probe_failure_attribution_v1',
 'created_at': '2026-06-14T18:32:31.383117+00:00',
 'profile': 'release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1',
 'decision': 'release16_full_probe_failure_attribution_no_actor_retention',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'soxlbacktest_reference': {'converted_qilib_daily_path': 'runtime/production/card/soxl_backtest_policy_reimplementation_v1/new_policy_daily.csv',
  'ytd_return_pct': 132.07797795135497,
  'ytd_max_drawdown_pct': 13.554088542803521,
  'legacy_alignment_delta_pct': 0.0},
 'release16_ytd': {'return_pct': 38.555494067096085,
  'relative_return_pp': -288.37080887525156,
  'max_drawdown_pct': 9.559552575516284,
  'avg_position': 0.08525600842690058},
 'baseline_gates_failed': ['soxlbacktest_reference_return',
  'current_best_alignment_floor'],
 'poor_window_cluster_count': 6,
 'dominant_failure_types': {'underparticipation': 6},
 'top_feature_groups': [{'feat

## Conclusion

This attribution card keeps promotion and actor retention blocked. The latest release16 profile is not failing because of a single threshold; it combines severe participation gaps versus SOXLBacktest/current-best YTD with walk-forward drawdown and negative-return failures. The next admissible direction is a state-conditioned multi-objective target/checkpoint screen, not another unchanged full PPO run.
